# 🧠 FaceKey Studio — Kaggle Dual T4 GPU Deep Learning Trainer
### Unleash 100% of Kaggle's Free 30h/Week GPU Power (2x NVIDIA T4 GPUs @ 500+ FPS)

> [!IMPORTANT]
> **CRITICAL KAGGLE SETTINGS (RIGHT SIDEBAR):**
> 1. **Accelerator**: Select **GPU T4 x2** (Allocates 2x NVIDIA Tesla T4 GPUs with 30 GB combined VRAM)
> 2. **Internet**: Toggle **Internet ON** (Required to pull dataset chunks from Hugging Face Hub)

**⚡ TWO OPERATIONAL MODES:**
- **Mode A (Live Interactive — 12 Hours Limit)**: Click *Run All* or run cells individually to monitor real-time losses, velocity metrics, and FPS.
- **Mode B (Headless Cloud Commit — 9 Hours Limit)**: Click **Save Version** (top right) ➔ Select **Save & Run All (Commit)**. Kaggle runs the training completely in the background on cloud GPUs — you can close your browser and turn off your computer! Every epoch auto-syncs weights to Hugging Face Hub.

**🛡️ BUILT-IN CRASH & RESUME SAFEGUARDS:**
- **Zero-Disk In-Memory Streaming**: Loads `.tar.gz` chunks directly in RAM, extracts facial motion arrays, and cleans up archives (uses <500 MB permanent disk space, avoiding Kaggle's 20 GB disk limit).
- **Interruption-Proof Cloud Sync**: Best and latest model weights (`checkpoint_best.pt`, `checkpoint_latest.pt`) auto-sync to Hugging Face Hub every epoch.
- **1-Click Auto-Resume**: If a session expires or crashes, restarting the notebook automatically pulls the latest checkpoint from Hugging Face and resumes training from the exact next epoch with optimizer state intact!
- **CUDA Memory Protection**: AMP FP16 + gradient clipping + automatic inter-epoch VRAM defragmentation prevents OOM errors.

## 💓 STEP 0: [OPTIONAL] BROWSER KEEP-ALIVE HEARTBEAT (FOR INTERACTIVE SESSIONS)
> Run this cell if training interactively in your browser to prevent tab timeouts during long epochs.

In [ ]:
# Browser Keep-Alive Heartbeat (Prevents idle disconnect)
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive(){
    console.log('[FaceKey Heartbeat] Session alive at ' + new Date().toLocaleTimeString());
}
setInterval(keepAlive, 60000);
'''))
print('[✓] Session Keep-Alive Heartbeat Activated!')


## ⚡ STEP 1: VERIFY DUAL NVIDIA T4 GPUs & TENSOR CORES

In [ ]:
# Verify Dual T4 GPU allocation and Tensor Core availability
!nvidia-smi

import torch
gpu_count = torch.cuda.device_count()
print(f'\n[+] PyTorch CUDA Available: {torch.cuda.is_available()}')
print(f'[+] Detected GPUs: {gpu_count}')
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f'    • GPU {i}: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM, {props.multi_processor_count} SMs)')
if gpu_count >= 2:
    print('[✓] DUAL T4 GPU ACCELERATION READY!')
elif gpu_count == 1:
    print('[!] Single GPU detected. (For maximum speed, switch Accelerator to GPU T4 x2 in Notebook options).')
else:
    print('[!] WARNING: No GPU detected! Set Accelerator to GPU T4 x2 in the right sidebar.')


## ⚡ STEP 2: CLONE & UPDATE FACEKEY REPOSITORY (SELF-HEALING)

In [ ]:
# Clone or pull latest repository code
import os
%cd /kaggle/working

APP_DIR = '/kaggle/working/FaceKeyAnimation'
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating FaceKeyAnimation to latest GitHub Gen2 branch...')
    !cd {APP_DIR} && git fetch --all --prune && git checkout Gen2 && git reset --hard origin/Gen2 && git clean -fd
else:
    print('[*] Fresh clone of FaceKeyAnimation (Gen2 branch) from GitHub...')
    !rm -rf {APP_DIR}
    !git clone -b Gen2 https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}
!pip install -q -U torch torchvision torchaudio huggingface_hub numpy scipy
print('[+] Gen2 Deep Learning Training environment ready with latest GitHub code!')


## ⚡ STEP 3: LAUNCH DUAL T4 GPU TRAINING (PHASE 1 BENCHMARK: 15 CHUNKS)

In [ ]:
# ==============================================================================
# 🚀 1-CLICK DUAL T4 GPU MODEL TRAINING (SPEECH-TO-FACIAL ANIMATION)
# ==============================================================================
import os

# 1. Resolve Hugging Face Credentials
HF_REPO = 'VijayTheOne/facekey-dataset-chunks'
HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass

if not HF_TOKEN and os.path.exists('/kaggle/working/hf_token.txt'):
    with open('/kaggle/working/hf_token.txt') as f:
        HF_TOKEN = f.read().strip()

if not HF_TOKEN:
    HF_TOKEN = input('Enter Hugging Face Token (starts with hf_): ').strip()
    if HF_TOKEN:
        with open('/kaggle/working/hf_token.txt', 'w') as f:
            f.write(HF_TOKEN)

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_REPO'] = HF_REPO

print(f'[+] Hugging Face Repo  : {HF_REPO}')
print(f'[+] Hugging Face Token : Configured ({len(HF_TOKEN)} chars)')

# 2. Launch Multi-GPU Training Engine
# - Automatically pulls 5 clean chunks (~2 GB) from Hugging Face Hub
# - Ultra-lean in-memory streaming compiles dataset directly in RAM (<200 MB RAM!)
# - Automatically cleans up temporary tar archives to keep disk usage near 0 MB
# - Runs DataParallel across Dual T4 GPUs with FP16 Tensor Cores
# - Saves best and latest checkpoints to Hugging Face Hub after every epoch!
%cd /kaggle/working/FaceKeyAnimation
!python scripts/train_speech_to_animation.py \
    --data-dir /kaggle/working/training_data \
    --checkpoints-dir /kaggle/working/checkpoints \
    --hf-repo "$HF_REPO" \
    --hf-token "$HF_TOKEN" \
    --max-chunks 5 \
    --epochs 50 \
    --batch-size 128 \
    --lr 1e-4 \
    --seq-len 64


---
## 📊 STEP 4: MONITOR GPU USAGE & VRAM (RUN WHILE TRAINING)

In [ ]:
# Real-time Dual GPU VRAM and Utilization snapshot
!nvidia-smi --query-gpu=index,name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu --format=csv


---
## 🚨 STEP 5: MANUAL RESCUE & SAVE CHECKPOINTS TO HUGGING FACE
> Run this cell anytime if training is stopped or before your Kaggle session finishes to ensure all saved weights are in your Hugging Face cloud repository.

In [ ]:
# Force sync checkpoints to Hugging Face Hub
import os, glob
from huggingface_hub import HfApi

ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
if not ckpts:
    print('[!] No .pt checkpoints found in /kaggle/working/checkpoints')
else:
    api = HfApi()
    print(f'[*] Uploading {len(ckpts)} checkpoint(s) to Hugging Face...')
    for ck in ckpts:
        fname = os.path.basename(ck)
        sz_mb = os.path.getsize(ck) / (1024 * 1024)
        print(f'  -> Uploading {fname} ({sz_mb:.1f} MB)...', end=' ', flush=True)
        api.upload_file(
            path_or_fileobj=ck,
            path_in_repo=f'checkpoints/{fname}',
            repo_id=HF_REPO,
            repo_type='dataset',
            token=HF_TOKEN
        )
        print('[DONE ✓]')
    print('[✓] All checkpoints safely secured in Hugging Face Hub!')
